# Boston Celtics 2025–26: Tactical Evaluation and Key Performance Drivers

Author: Daniel López

Dataset: 2025–26 NBA Regular Season Statistics (Player & Team Data)

# Executive Summary & Analytical Framework


This report presents a quantitative tactical evaluation of the **2025–26 Boston Celtics** roster, analyzing player efficiency, lineup synergy, and roster architecture using traditional metrics, advanced on/off ratings, and unsupervised machine learning (**PCA + K-Means Clustering**). The analysis evaluates regular-season performance drivers and outlines strategic priorities following major off-season roster restructuring.


### Executive Summary

#### Key Findings & Tactical Bottlenecks
* **Primary Star Impact & Load Allocation:** **Jayson Tatum** anchored the team's statistical baseline upon returning from injury, leading in both rebounds and assists while ranking Top 5 across four primary traditional metrics. In his absence, **Jaylen Brown** carried the primary scoring load ($28.7\text{ PTS/G}$).
* **Backcourt Efficiency Regression:** The primary offensive drag stemmed from backcourt shooting drops: **Payton Pritchard** regressed from $63.3\%$ to $58.4\%\text{ TS\%}$ ( -4.9\%), while **Derrick White** fell to $52.9\%\text{ TS\%}$ (-7.7%). Despite his scoring efficiency drop, White remained the team's definitive two-way connector, leading in both steals and blocks.
* **The Center Dilemma & Spacing:** **Neemias Queta** established himself as Boston's most efficient interior anchor (leading in `REB/G`, `FG%`, `TS%`, and a roster-high $+13.2\text{ Net Rating}$). However, his lack of perimeter range forced Coach Joe Mazzulla to balance 5-out spacing requirements against **Luka Garza's** perimeter threat ($43.3\%\text{ 3FG\%}$) and defensive trade-offs.
* **High-Impact Rotation Value:** Young guard **Hugo González** logged the 2nd-highest `Net Rating` ($+11.9$) and a $106.2\text{ Defensive Rating}$ on limited minutes, emerging as a premier bench 3-and-D candidate[cite: 1].
* **Unsupervised Archetypes (PCA + K-Means):** A 2-component PCA captured **$66.1\%$ of total variance**, categorizing the 11 qualified rotation players into four functional roles: *Primary Scorers*, *Playmakers*, *Floor Spacers*, and *Defensive Anchors*.

### Analytical Framework & Metrics Glossary

To ensure clarity for both technical and executive stakeholders, the metrics utilized throughout this report are defined below:

| Metric Category | Variable | Full Name | Definition / Formula Concept | Tactical Relevance |
| :--- | :--- | :--- | :--- | :--- |
| **Traditional** | `PTS` / `REB` / `AST` | Points / Rebounds / Assists | Standard box-score counting stats per game. | Evaluates basic production volume. |
| **Traditional** | `STL` / `BLK` | Steals / Blocks | Defensive disruptive plays per game. | Measures individual event-based defensive impact. |
| **Shooting** | `FG%` / `3FG%` | Field Goal % / 3-Point % | Unadjusted shot conversion rates. | Baseline shooting accuracy across ranges. |
| **Advanced** | `OFF_RATING` | Offensive Rating | Points scored by team per 100 possessions on court. | Individual's overall offensive efficiency on court. |
| **Advanced** | `DEF_RATING` | Defensive Rating | Points allowed by team per 100 possessions on court. | Individual's overall defensive containment (*lower is better*). |
| **Advanced** | `NET_RATING` | Net Rating | $\text{Offensive Rating} - \text{Defensive Rating}$ | Total point differential impact per 100 possessions. |
| **Advanced** | `TS_PCT` (TS%) | True Shooting % | $\frac{\text{PTS}}{2 \times (\text{FGA} + 0.44 \times \text{FTA})}$ | Comprehensive efficiency adjusting for 3-pointers and free throws. |
| **Advanced** | `USG_PCT` (USG%) | Usage Rate | % of team plays used by player while on court. | Measures offensive load and play-ending volume. |
| **Advanced** | `PIE` | Player Impact Estimate | Measure of a player's overall statistical game contribution share. | Single-metric holistic box-score valuation. |

## Table of content
Section 1: Code

Section 2: Top 5 players from Advanced and Traditional Stats

Section 3: Optimal Lineup Configurations & Role Optimization

Section 4: Dimensionality Reduction & Roster Clustering (PCA + K-Means) to find the roles in the team

Section 5: Offensive Load vs. Efficiency Matrix (USG% vs. TS%)

Section 6: Executive Conclusions & Front Office Insights

## Code

Libraries

In [1]:
import pandas as pd 
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from scipy.optimize import linear_sum_assignment
import plotly.express as px
import plotly.io as pio
pio.renderers.default = "notebook_connected"

Data importation

In [2]:
Ja = pd.read_csv("celtics_data/players_advanced.csv")
Jn = pd.read_csv("celtics_data/players_traditional.csv")
Ea = pd.read_csv("celtics_data/team_advanced.csv")
En = pd.read_csv("celtics_data/team_traditional.csv")

Data Processing

In [3]:
STATS_TRAD = ["PTS", "REB", "AST", "STL", "BLK", "FG_PCT", "FG3_PCT"]
PERCENT_STATS_TRAD = {"FG_PCT", "FG3_PCT"}
LABELS_TRAD = {
    "PTS": "PTS per Game",
    "REB": "REB per Game",
    "AST": "AST per Game",
    "STL": "STL per Game",
    "BLK": "BLK per Game",
    "FG_PCT": "FG%",
    "FG3_PCT": "3FG%",
}

top_all_traditional = {
    col: Jn.nlargest(5, col)[["PLAYER_NAME", col]] for col in STATS_TRAD
}

N_COLS = 2
N_ROWS = -(-len(STATS_TRAD) // N_COLS)

fig_traditional = make_subplots(
    rows=N_ROWS,
    cols=N_COLS,
    subplot_titles=[LABELS_TRAD[s] for s in STATS_TRAD],
    horizontal_spacing=0.15,
    vertical_spacing=0.10,
)

for i, stat in enumerate(STATS_TRAD):
    row = i // N_COLS + 1
    col = i % N_COLS + 1

    df_stat = top_all_traditional[stat].sort_values(stat, ascending=True)

    if stat in PERCENT_STATS_TRAD:
        text_labels = [f"{v:.1%}" for v in df_stat[stat]]
    else:
        text_labels = [f"{v:.1f}" for v in df_stat[stat]]

    fig_traditional.add_trace(
        go.Bar(
            x=df_stat[stat],
            y=df_stat["PLAYER_NAME"],
            orientation="h",
            text=text_labels,
            textposition="outside",
            marker_color="#007A33",
            showlegend=False,
        ),
        row=row,
        col=col,
    )

    max_val = df_stat[stat].max()
    fig_traditional.update_xaxes(
        range=[0, max_val * 1.25], row=row, col=col, showticklabels=False
    )
    fig_traditional.update_yaxes(row=row, col=col, automargin=True)

Advanced_Stats = ["OFF_RATING", "DEF_RATING", "NET_RATING", "TS_PCT"]
LABELS_ADV = {
    "OFF_RATING": "Offensive Rating",
    "DEF_RATING": "Defensive Rating",
    "NET_RATING": "Net Rating",
    "TS_PCT": "True Shooting %",
}
HIGHER_IS_BETTER = {
    "OFF_RATING": True,
    "DEF_RATING": False,
    "NET_RATING": True,
    "TS_PCT": True,
}
PERCENT_STATS_ADV = {"TS_PCT"}

top_all_advanced = {}
for col in Advanced_Stats:
    if HIGHER_IS_BETTER[col]:
        top_all_advanced[col] = Ja.nlargest(5, col)[["PLAYER_NAME", col]]
    else:
        top_all_advanced[col] = Ja.nsmallest(5, col)[["PLAYER_NAME", col]]

N_COLS = 2
N_ROWS = -(-len(Advanced_Stats) // N_COLS)

fig_advanced = make_subplots(
    rows=N_ROWS,
    cols=N_COLS,
    subplot_titles=[LABELS_ADV[s] for s in Advanced_Stats],
    horizontal_spacing=0.15,
    vertical_spacing=0.10,
)

for i, stat in enumerate(Advanced_Stats):
    row = i // N_COLS + 1
    col = i % N_COLS + 1

    ascending = HIGHER_IS_BETTER[stat]
    df_stat = top_all_advanced[stat].sort_values(stat, ascending=ascending)

    if stat in PERCENT_STATS_ADV:
        text_labels = [f"{v:.1%}" for v in df_stat[stat]]
    else:
        text_labels = [f"{v:.1f}" for v in df_stat[stat]]

    fig_advanced.add_trace(
        go.Bar(
            x=df_stat[stat],
            y=df_stat["PLAYER_NAME"],
            orientation="h",
            text=text_labels,
            textposition="outside",
            marker_color="#007A33",
            showlegend=False,
        ),
        row=row,
        col=col,
    )

    max_val = df_stat[stat].max()
    if HIGHER_IS_BETTER[stat]:
        fig_advanced.update_xaxes(
            range=[0, max_val * 1.25], row=row, col=col, showticklabels=False
        )
    else:
        fig_advanced.update_xaxes(
            range=[0, max_val * 1.15], row=row, col=col, showticklabels=False
        )

    fig_advanced.update_yaxes(row=row, col=col, automargin=True)


MIN_GP = 16
MIN_MPG = 12

Ja_filtered = Ja[(Ja["GP"] >= MIN_GP) & (Ja["MIN"] >= MIN_MPG)].copy()

def make_top10_table(df, stat, title, ascending, value_fmt="{:.1f}"):
    """Builds a Top 10 table figure and returns it as a Plotly object (unrendered)."""
    cols = ["PLAYER_NAME", "GP", "MIN", stat]
    df_top = (
        df.sort_values(stat, ascending=ascending)
        .head(10)[cols]
        .reset_index(drop=True)
    )
    df_top.insert(0, "Rank", range(1, len(df_top) + 1))

    gp_formatted = df_top["GP"].astype(int).astype(str).tolist()
    min_formatted = [f"{v:.1f}" for v in df_top["MIN"]]
    stat_formatted = [value_fmt.format(v) for v in df_top[stat]]

    n = len(df_top)
    row_colors = ["#D9F2E6" if i < 5 else "white" for i in range(n)]

    fig_table = go.Figure(
        data=[
            go.Table(
                columnwidth=[50, 220, 60, 80, 110],
                header=dict(
                    values=["#", "Player", "GP", "MIN", title],
                    fill_color="#007A33",
                    font=dict(color="white", size=13, family="Arial"),
                    align="center",
                    height=32,
                ),
                cells=dict(
                    values=[
                        df_top["Rank"],
                        df_top["PLAYER_NAME"],
                        gp_formatted,
                        min_formatted,
                        stat_formatted,
                    ],
                    fill_color=[row_colors] * 5,
                    font=dict(color="black", size=12, family="Arial"),
                    align=["center", "left", "center", "center", "center"],
                    height=28,
                ),
            )
        ]
    )

    fig_table.update_layout(
        title_text=f"Boston Celtics — Top 10 {title} (2025-26)",
        title_x=0.5,
        template="plotly_white",
        width=650,
        height=430,
        margin=dict(t=60, l=10, r=10, b=10),
    )
    return fig_table


table_def = make_top10_table(
    Ja_filtered, "DEF_RATING", "Defensive Rating", ascending=True
)
table_off = make_top10_table(
    Ja_filtered, "OFF_RATING", "Offensive Rating", ascending=False
)
table_net = make_top10_table(
    Ja_filtered, "NET_RATING", "Net Rating", ascending=False
)

MIN_FG3A = 1.5
Jn_filtered = Jn[Jn["FG3A"] >= MIN_FG3A].copy()

table_fg3 = make_top10_table(
    Jn_filtered, "FG3_PCT", "3PT %", ascending=False, value_fmt="{:.1%}"
)

# Note: table_def, table_off, table_net, and table_fg3 are stored objects, this were made to select the players for the starting 5s

df_pca = Ja.merge(
    Jn[["PLAYER_ID", "FGA"]],
    on="PLAYER_ID",
    how="left",
).rename(columns={"FGA": "FGA_PG"})

df_pca = df_pca[
    (df_pca["GP"] >= MIN_GP) & (df_pca["MIN"] >= MIN_MPG)
].reset_index(drop=True)

# Define feature scope by role
ROLE_VARS = {
    "Defensive Anchor": ["DREB_PCT", "REB_PCT", "DEF_RATING"],
    "Floor Spacer / Efficiency": ["EFG_PCT", "TS_PCT", "OFF_RATING"],
    "Playmaker": ["AST_PCT", "AST_TO", "AST_RATIO"],
    "Primary Scorer": ["USG_PCT", "FGA_PG", "PIE"],
}

FEATURE_COLS = list(
    dict.fromkeys(col for cols in ROLE_VARS.values() for col in cols)
)

# Standardize metrics & invert Defensive Rating direction
X_raw = df_pca[FEATURE_COLS].copy()
scaler = StandardScaler()
X_scaled = pd.DataFrame(
    scaler.fit_transform(X_raw), columns=FEATURE_COLS, index=df_pca.index
)
X_scaled["DEF_RATING"] = -X_scaled["DEF_RATING"]

# Fit PCA (2 Components)
pca = PCA(n_components=2, random_state=42)
pca_coords = pca.fit_transform(X_scaled)
df_pca["PC1"] = pca_coords[:, 0]
df_pca["PC2"] = pca_coords[:, 1]
var_exp = pca.explained_variance_ratio_

# Fit KMeans (k=4)
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
df_pca["Cluster"] = kmeans.fit_predict(X_scaled)

# Calculate composite scores and mapping
for role, cols in ROLE_VARS.items():
    df_pca[f"score_{role}"] = X_scaled[cols].mean(axis=1)

cluster_ids = sorted(df_pca["Cluster"].unique())
score_matrix = np.array([
    [
        df_pca.loc[df_pca["Cluster"] == c, f"score_{role}"].mean()
        for role in ROLE_VARS
    ]
    for c in cluster_ids
])

row_ind, col_ind = linear_sum_assignment(-score_matrix)
roles_list = list(ROLE_VARS.keys())
cluster_to_role = {
    cluster_ids[r]: roles_list[c] for r, c in zip(row_ind, col_ind)
}
df_pca["Role"] = df_pca["Cluster"].map(cluster_to_role)

# Build Top 3 by archetype
top3_by_role = {}
for role in ROLE_VARS:
    subset = df_pca[df_pca["Role"] == role].copy()
    top3_by_role[role] = subset.sort_values(
        f"score_{role}", ascending=False
    ).head(3)

# Build Scatter Plot object
fig_pca = px.scatter(
    df_pca,
    x="PC1",
    y="PC2",
    color="Role",
    text="PLAYER_NAME",
    title="Boston Celtics — Functional Player Profiles (PCA + KMeans, 2025-26)",
    template="plotly_white",
    width=900,
    height=650,
)

usg_mean = Ja_filtered["USG_PCT"].mean()
ts_mean = Ja_filtered["TS_PCT"].mean()

fig_usage_efficiency = px.scatter(
    Ja_filtered,
    x="USG_PCT",
    y="TS_PCT",
    text="PLAYER_NAME",
    size="MIN",
    size_max=22,
    color_discrete_sequence=["#007A33"],
    title="Boston Celtics — Usage Rate vs. True Shooting (2024-25)",
    labels={"USG_PCT": "Usage % (USG%)", "TS_PCT": "True Shooting % (TS%)"},
    template="plotly_white",
    width=900,
    height=700,
)

### Section 1: Top 5 players from Advanced and Traditional Stats

In [4]:
fig_traditional.update_layout(
    title_text="Boston Celtics — Top 5 Leaders in Traditional Statistics",
    title_x=0.5,
    template="plotly_white",
    height=1100,
    width=950,
    font=dict(family="Arial", size=13, color="black"),
    margin=dict(t=100, l=20, r=20, b=20),
)

Despite returning from an 8-month injury, Jayson Tatum ranks in the Top 5 across 4 of the 5 primary traditional statistical categories while leading the roster in both rebounds and assists. In his absence, Jaylen Brown carried the offensive load, leading the team in scoring (PTS/G) and appearing in 4 of the 5 core categories.

The clear X-factor for this roster is Derrick White, the only player to appear in every single Top 5 traditional metric while leading the team in both steals and blocks per game—epitomizing the versatile profile required of a championship contender.

Complementing the core, Neemias Queta provides high-efficiency rim protection and rebounding (leading in REB/G and FG%, ranking 2nd in BLK/G), while Payton Pritchard stepped up as the primary point guard, contributing as a floor-spacing scorer (3rd in PTS/G, Top 5 in AST and 3FG%) and validating his 6MOTY pedigree.
fig_advanced.update_layout(
    title_text="Boston Celtics — Top 5 Leaders in Advanced Statistics (2025-26)",
    title_x=0.5,
    template="plotly_white",
    height=650,
    width=950,
    font=dict(family="Arial", size=13, color="black"),
    margin=dict(t=100, l=20, r=20, b=20),
)
fig_advanced.add_annotation(
    text="* For Defensive Rating, a lower value indicates superior defensive containment",
    xref="paper",
    yref="paper",
    x=0.5,
    y=-0.08,
    showarrow=False,
    font=dict(size=11, color="gray"),
)

In [5]:
fig_advanced.add_annotation(
    text="* For Defensive Rating, lower value = better defense",
    xref="paper", yref="paper",
    x=0.5, y=-0.08,
    showarrow=False,
    font=dict(size=11, color="gray"),
)

A critical takeaway from the rating distribution is that no single player ranks in the top tiers for both Offensive Rating (ORtg) and Defensive Rating (DRtg). This separation highlights a lack of elite two-way production in the top metrics—a potential vulnerability in playoff environments where multi-dimensional versatility is essential.

Despite this split, Neemias Queta emerges as a major statistical standout. He leads the team in Net Rating, ranks second in Defensive Rating, and tops the roster in True Shooting Percentage (TS%), solidifying his role as the team's most efficient interior option.

Additionally, Sam Hauser's offensive value is clearly displayed: while absent from the defensive leaderboards, his elite perimeter shooting makes him one of the primary beneficiaries of Joe Mazzulla's heavy five-out offensive scheme.

### Optimal Lineup Configurations & Role Optimization

In [6]:
print(
    f"Qualified players (rating tables): {len(Ja_filtered)} of {len(Ja)} "
    f"(filters: GP >= {MIN_GP}, MPG >= {MIN_MPG})"
)
print(
    f"Qualified players (3FG% table): {len(Jn_filtered)} of {len(Jn)} "
    f"(filter: 3PA >= {MIN_FG3A} per game)"
)

Qualified players (rating tables): 11 of 17 (filters: GP >= 16, MPG >= 12)
Qualified players (3FG% table): 13 of 17 (filter: 3PA >= 1.5 per game)


To eliminate statistical noise from garbage-time possessions, qualified players were filtered to a minimum of 16 Games Played ($GP \ge 16$) and 12 Minutes per Game ($MPG > 12$). This reduces the evaluation pool from 17 to the 11 primary rotation contributors (accounting for Jayson Tatum’s 17-game sample following his recovery).

Optimal Defensive Lineup (Defensive Rating)

Filtering for defensive containment yields a natural 1-through-5 positional fit that mirrors the top 5 individual Defensive Ratings ($DRtg$) on the roster:

| Position | Player | DRtg (Rank) |
| --- | --- | --- |
| **PG** | Derrick White | 108.0 (3rd) |
| **SG** | Hugo González | 106.2 (2nd) |
| **SF** | Baylor Scheirman | 109.0 (4th) |
| **PF** | Jayson Tatum | 110.2 (5th) |
| **C** | Neemias Queta | 105.6 (1st) |

* **Tactical Insight:** This unit represents the team's absolute defensive ceiling. With multi-positional perimeter switching from White and González paired with Queta’s elite interior protection, this lineup minimizes high-value opponent scoring opportunities.

Optimal Offensive Lineup (Offensive Rating)

Selecting the top 5 offensive efficiency ratings ($ORtg$) also naturally forms a balanced, high-octane 5-man unit:

| Position | Player | ORtg (Rank) |
| --- | --- | --- |
| **PG** | Payton Pritchard | 120.1 (2nd) |
| **SG** | Jaylen Brown | 119.5 (4th) |
| **SF** | Sam Hauser | 119.8 (3rd) |
| **PF** | Jayson Tatum | 120.7 (1st) |
| **C** | Luka Garza | 119.4 (5th) |

* **Tactical Insight:** Built to maximize scoring output per 100 possessions, this unit pairs elite star usage (Tatum & Brown) with elite perimeter gravity (Hauser) and interior scoring efficiency (Garza).

Pure Perimeter Spacing Lineup ("Mazzulla Ball")

Applying a volume threshold of at least **1.5 3-Point Attempted per game ($3PA \ge 1.5$)** isolates the ultimate floor-spacing lineup:

| Position | Player | 3FG% (Rank) |
| --- | --- | --- |
| **PG** | Payton Pritchard | 37.7% (5th) |
| **SG** | Baylor Scheirman | 39.9% (2nd) |
| **SF** | Jordan Walsh | 38.4% (4th) |
| **PF** | Sam Hauser | 39.3% (3rd) |
| **C** | Luka Garza | 43.3% (1st) |

* **Tactical Insight:** This lineup represents the pinnacle of Coach Joe Mazzulla's perimeter-heavy philosophy. **Payton Pritchard** and **Sam Hauser** stand out as the primary offensive bridges, appearing in both the top Offensive and 3-Point Shooting units despite Hauser primarily operating off the bench.

The Ideal Starting 5 (Net Rating Differential)


Balancing offensive production with defensive suppression ($\Delta = \text{ORtg} - \text{DRtg}$) reveals the most impactful 5-man combination on the roster:

| Position | Player | Net Rating (Rank) |
| --- | --- | --- |
| **PG** | Derrick White | +11.3 (3rd) |
| **SG** | Hugo González | +11.9 (2nd) |
| **SF** | Sam Hauser | +8.8 (5th) |
| **PF** | Jayson Tatum | +10.5 (4th) |
| **C** | Neemias Queta | +13.2 (1st) |

* **Critical Takeaways & Strategic Recommendations:**
1. **The Hugo González Surge:** Despite logging the lowest total minutes among qualified rotation players, rookie/young guard Hugo González ranks 2nd overall in Net Rating (+11.9). Increasing his minutes allocation next season is a primary recommendation to solidify perimeter containment.
2. **The Jaylen Brown On/Off Anomaly:** Despite carrying the primary offensive burden, Jaylen Brown ranks 8th out of 11 in individual Net Rating. This reflects heavy usage against elite opponent starting units without consistent two-way lineup support, reinforcing the need to pair him with high-efficiency glue guys like White and Queta.



### Dimensionality Reduction & Roster Clustering (PCA + K-Means) to find the roles in the team

By applying **Principal Component Analysis (PCA)** across 11 primary rotation players and 12 advanced statistical metrics (`DREB%`, `REB%`, `DEF_RATING`, `EFG%`, `TS%`, `OFF_RATING`, `AST%`, `AST_TO`, `AST_RATIO`, `USG%`, `FGA_PG`, `PIE`), we reduced the dataset's complexity to two primary dimensions while capturing **66.1% of the total variance**:

* **PC1 (42.8% Variance — Offensive Volume & Usage):** Differentiates high-usage decision-makers from off-ball role players.
* **PC2 (23.3% Variance — Interior Impact vs. Perimeter Playmaking):** Distinguishes interior rebounding profiles from perimeter-oriented ball handlers.

In [7]:
fig_pca.update_traces(textposition="top center", marker=dict(size=10))

### Quadrant Breakdown & Tactical Findings

* **Top-Right (High Usage + Interior/Scoring Impact):** **Jayson Tatum** anchors this quadrant as an elite primary scorer who also contributes heavily on the glass. **Jaylen Brown** and **Nikola Vučević** join him as primary offensive options who absorb significant scoring responsibility.
* **Bottom-Right (High Usage + Perimeter Playmaking):** **Derrick White** and **Payton Pritchard** populate this area, defined by high ball-handling involvement, low turnover rates, and elite distribution metrics.
* **Bottom-Left (Low Usage + Perimeter Spacing):** **Sam Hauser** and **Baylor Scheirman** operate as off-ball floor spacers. While primarily known for their perimeter shooting, their structural placement points toward development into versatile "3-and-D" wing options.
* **Top-Left (Low Usage + High Interior Efficiency):** **Neemias Queta** and **Luka Garza** represent high interior efficiency with minimal ball dominance, maximizing their field goal percentage without consuming team usage.
* **Defensive Upside Candidates:** **Hugo González** and **Jordan Walsh** sit near the defensive/efficiency boundary with low total usage. Given their positive defensive impact per possession, expanding their minutes and shot attempts next season is a strategic recommendation.

## Offensive Load vs. Efficiency Matrix (USG% vs. TS%)

In [8]:
# Reference lines: team averages, splitting the chart into 4 quadrants
fig_usage_efficiency.add_vline(x=usg_mean, line_dash="dash", line_color="gray")
fig_usage_efficiency.add_hline(y=ts_mean, line_dash="dash", line_color="gray")
fig_usage_efficiency.show()

### TS% Benchmark Framework & Tactical Analysis

**Standardized Efficiency Scale (TS%)**

* **$\le 53.0\%$ (Sub-Threshold / Inefficient):** Severe shooting struggles; creates negative offensive value per possession.
* **$53.0\% - 57.0\%$ (Below Average):** Non-disastrous, but below league-standard scoring efficiency.
* **$57.0\% - 59.0\%$ (NBA Average):** Functional baseline; adequate spacing presence without drawing focused defensive coverage.
* **$59.0\% - 62.0\%$ (Above Average / Good):** High-level efficiency; characteristic of primary 3-point threats or strong interior finishers.
* **$\ge 62.0\%$ (Elite / Hyper-Efficient):** High-volume rim finishers or elite primary creators with exceptional shot selection.

---

### The 2025–26 Efficiency Paradox

* **The Center Dilemma (Queta vs. Garza):** Joe Mazzulla’s 5-out tactical scheme faced a structural conflict at center. **Neemias Queta** delivered elite interior efficiency ($\ge 62\%$ TS%), offensive rebounding, and rim protection, but provided zero perimeter threat. Conversely, **Luka Garza** offered theoretical 3-point spacing but lacked the defensive containment required for sustained starting minutes, forcing Mazzulla to rely on Queta and demand perimeter volume elsewhere.
* **Underutilized Perimeter Spacers:** High-efficiency spot-up threats—**Sam Hauser**, **Baylor Scheirman**, and **Jordan Walsh**—were unable to absorb the missing perimeter volume due to limited minute allocations and lower usage roles.
* **Guard Efficiency Regression (Year-over-Year Drop):** The primary driver of Boston's perimeter friction was the simultaneous efficiency drop across the primary backcourt:
* **Payton Pritchard:** Regressed from $63.3\%$ TS% (Elite) down to $58.4\%$ ( -4.9\%$), shifting from a premium hyper-efficient scorer to league-average production.
* **Derrick White:** Experienced a major regression from $60.6\%$ TS% down to $52.9\%$ ($ -7.7\%$), falling into the sub-$53\%$ threshold. While White maintained elite multi-category defensive impact, his scoring conversion became a noticeable offensive drag.



---

### Integration into Section 5 (USG% vs. TS% Matrix)

This efficiency drop explains why Boston's offensive rating experienced volatility despite high overall usage. In Section 5, plotting these exact regressions on the **USG% vs. TS% Scatter Plot** will visually isolate:

1. **High-Usage / Low-Efficiency Drag:** Derrick White's position in the bottom-right quadrant.
2. **Efficiency Bottlenecks:** Queta's high TS% isolated on low usage / zero perimeter distance.
3. **Target Re-allocation Candidates:** Identifying how transferring 2–3 field goal attempts per game to Hauser/Scheirman stabilizes team True Shooting without altering core playmaking roles.

# Executive Conclusions & Front Office Insights

### Off-Season Restructuring & Strategic Rationale

* **Wing Re-Tooling: Jaylen Brown Transferred for Paul George & 2 Future 1st-Round Picks**

 **Financial & Asset Runway:** Trading Brown shifts significant multi-year financial commitments into shorter-term salary flexibility while adding valuable draft capital.

 **Perimeter Spacing Upgrade:** George recorded 17.0 PPG on 39.2% 3-point shooting. In Coach Joe Mazzulla's high-volume, shot-quality-generating offense, George's perimeter conversion is projected to eclipse 40.0%, providing elite spot-up and secondary creation with a reduced usage demand.


* **Frontcourt Anchoring: Acquisition of Mitchell Robinson**

 **Interior Dominance:** Robinson brings elite efficiency (67.6% TS%) and rebounding (8.8 RPG). His 106 Defensive Rating would have ranked second overall on Boston's 2025–26 roster.
 
 **Tactical Evolution (The Double-Big Return):** While Robinson offers zero 3-point range, pairing him with Neemias Queta gives Boston two elite interior anchors. This shifts Boston from an exclusive 5-out model toward a hybrid defensive system reminiscent of the 2022 Finals run (Horford/Williams), providing necessary physical containment against elite interior stars across the league.


* **Backcourt Stabilization: Addition of Mike Conley**

 **Mentorship & Rotation Balance:** Conley absorbs high-stress playmaking possessions, reducing the creation burden on White and allowing Pritchard to return to a hyper-efficient off-ball microwave scoring role.

### Tactical & Operational Priorities for 2026–27

* **Implement Hybrid Defensive Schemes:** Alternate dynamically between Mazzulla's signature 5-out spacing and dual-anchor rim protection (Mitchell Robinson + Neemias Queta) based on matchup requirements rather than forcing a single tactical framework.
* **Re-allocate Shot Distribution:** Shift secondary usage away from regressed perimeter volume toward high-efficiency wing threats (**Paul George**, **Sam Hauser**, **Baylor Scheirman**).
* **Develop Bench 3-and-D Prospects:** Expand rotational minutes for **Hugo González**, **Jordan Walsh**, and **Baylor Scheirman**. Evaluating their production under higher workloads during the regular season will isolate Boston's premier bench 3-and-D option ahead of the playoffs.

### Trade Deadline Contingencies (February 2027)

* **Derrick White Salary Re-evaluation:** If Derrick White's scoring efficiency fails to bounce back toward his historic baselines, his $30M+ annual contract becomes a primary asset for salary restructuring before the trade deadline.
* **Targeting Stretch-Big Archetypes:** To optimize Coach Mazzulla’s 5-out philosophy without forfeiting defensive containment, explore trade markets for versatile bigs (PF/C) who offer legitimate perimeter range alongside competent defense targeting archetypes similar to **Santi Aldama** or **Onyeka Okongwu**.

### Executive Outlook & Competitive Horizon

* **Leadership Transition & Roster Ceiling:** While Jaylen Brown anchored the offense during a transitional campaign, the return of a fully healthy **Jayson Tatum** as the undisputed primary option—supported by elite perimeter shooting (Paul George), interior rim protection (Mitchell Robinson), and veteran poise (Mike Conley)—elevates Boston's baseline depth and structural ceiling.
* **Eastern Conference Projection:** After securing the **#2 seed in the Eastern Conference** during a transition-heavy 2025–26 regular season, Boston’s upgraded depth, enhanced shooting, and veteran additions position the franchise to project firmly within the **1st to 3rd seed range** for 2026–27.